In [2]:
# ---------------------------------------------------------------------------
# SETUP OG IMPORTS
# ---------------------------------------------------------------------------
from pathlib import Path
import sys
import numpy as np
from sympy import *

candidates = [Path.cwd(), *Path.cwd().parents]
SUPPORT_ROOT = next(
    (
        p / "support_material"
        for p in candidates
        if (p / "support_material" / "scripts" / "control").exists()
    ),
    None,
)
if SUPPORT_ROOT is None:
    SUPPORT_ROOT = next((p for p in candidates if (p / "scripts" / "control").exists()), None)
if SUPPORT_ROOT is None:
    raise RuntimeError("Kan ikke finde support_material/scripts/control.")
if str(SUPPORT_ROOT) not in sys.path:
    sys.path.insert(0, str(SUPPORT_ROOT))

from scripts.control import (
    analyze_transfer_function,
    bode_to_transfer,
    closed_loop_characteristic,
    closed_loop_poles,
    design_pi_lead_at_crossover,
    evaluate_transfer_function,
    find_stable_gain_ranges,
    ideal_disturbance_feedforward,
    phase_margin_from_point,
    second_order_characteristics,
    solve_lag_beta,
    solve_stability_interval_by_boundary,
    transfer_function_poles,
    unity_feedback_step_error,
)

print(f"Klar. Supportmateriale: {SUPPORT_ROOT}")

s = symbols("s")
K = symbols('K')
X1, X2, F = symbols("X1 X2 F")


Klar. Supportmateriale: c:\Users\sehes\GitHub\2026-spring-formelsamlinger\Linear control design 1\support_material


In [3]:
# ---------------------------------------------------------------------------
# TRANSFERFUNKTIONER, POLER, FEEDBACK OG RESPONS
# ---------------------------------------------------------------------------
# evaluate_transfer_function(numerator, denominator, omega)
# Input: koefficientlister i faldende potenser af s og frekvens omega [rad/s].
# Brug naar: beregn G(j*omega) fra en kendt transferfunktion.
# Eksempel: 1/(s+1) ved omega=1
# evaluate_transfer_function([1], [1, 1], 1)
#
# transfer_function_poles(denominator)
# Input: naevnerkoefficienter i faldende potenser af s.
# Brug naar: find poler efter at transferfunktionen er udledt.
# Eksempel: 1/(s+1)^2
# transfer_function_poles([1, 2, 1])
#
# analyze_transfer_function(G, variable=s)
# Input: symbolsk transferfunktion G(s), eventuelt med parameteren K.
# Returnerer: poler, nulpunkter, DC gain, orden, type og stabilitetsdata.
# Brug naar: en udledt transferfunktion skal analyseres symbolsk samlet.
# Eksempel: K = symbols("K", positive=True); analyze_transfer_function(K/(s**2 + 5*s + K), s)
#
# closed_loop_poles(numerator, denominator, proportional_gain=1.0, feedback_gain=1.0)
# Input: G(s)=num/den samt gain for et negativt feedback-loop.
# Brug naar: kontroller stabilitet for et foreslaaet P-gain.
# Eksempel: F21 Q11, G(s)=120/(s^3+43*s^2+120*s), Kp=25
# closed_loop_poles([120], [1, 43, 120, 0], proportional_gain=25)
#
# closed_loop_characteristic(forward, feedback=1, variable=s, negative_feedback=True)
# Input: symbolsk forward- og feedback-transferfunktion samt feedbackfortegn.
# Brug naar: udled karakteristisk polynomium foer stabilitetsanalyse.
# Eksempel: K = symbols("K", real=True); closed_loop_characteristic(K/(s*(s+5)), variable=s)
#
# unity_feedback_step_error(numerator, denominator, proportional_gain=1.0)
# Input: G(s)=num/den og Kp for E/R=1/(1+Kp*G).
# Brug naar: stationaer fejl for unit step i verificeret negativ unity feedback.
# Eksempel: F21 Q16
# unity_feedback_step_error([1224], [1, 30, 257, 612], proportional_gain=2)
#
# second_order_characteristics(denominator)
# Input: andenordensnaevner [a2, a1, a0] for a2*s^2+a1*s+a0.
# Returnerer: omega_n, zeta og procent overshoot.
# Brug naar: standard andenordens steprespons.
# Eksempel: F21 Q9 ved K=20
# second_order_characteristics([1, 5, 20])

# ---------------------------------------------------------------------------
# BODE, NYQUIST OG CONTROLLERDESIGN
# ---------------------------------------------------------------------------
# bode_to_transfer(dc_gain_db, poles=None, zeros=None, variable=s)
# Input: DC-gain i dB samt positive pol- og nulpunkt-knaekfrekvenser [rad/s].
# Brug naar: opstil G(s) fra et afleaest Bode-asymptotediagram.
# Eksempel: bode_to_transfer(20, poles=[10, 150], zeros=[100], variable=s)
#
# phase_margin_from_point(real_part, imaginary_part)
# Input: Nyquist-punkt paa enhedscirklen ved gain crossover.
# Brug naar: beregn phase margin fra et numerisk afleaest punkt.
# Eksempel: F21 Q14
# phase_margin_from_point(0.134, -0.99)
#
# design_pi_lead_at_crossover(numerator, denominator, omega_c, phase_margin_deg, n_i)
# Input: G(s)=num/den, valgt crossover, oensket phase margin og Ni=omega_c*tau_i.
# Returnerer: alpha, tau_i, tau_d, Kp og target-checks.
# Brug naar: PI-Lead-strukturen og specifikationerne allerede er valgt.
# Eksempel: F21 Q18
# den = np.polymul(np.polymul([5, 1], [1, 0.2, 0.6]), [0.01, 1])
# design_pi_lead_at_crossover([0.7, 0.35], den, 10, 45, 8)
#
# solve_lag_beta(lag_phase_deg, n_i)
# Input: kraevet negativ Lag-fase i grader og Ni=omega_c*tau_i.
# Brug naar: beta skal findes i et P-Lead-Lag design.
# Eksempel: F21 Q17
# solve_lag_beta(-8.9193, 3)
#
# ideal_disturbance_feedforward(plant_numerator, plant_denominator,
#                               disturbance_numerator, disturbance_denominator,
#                               disturbance_sign)
# Input: G(s), D(s) og disturbancefortegn (+1 eller -1) fra blokdiagrammet.
# Returnerer: Fd(s)-koefficienter samt proper/stable-status.
# Brug naar: nominal dynamisk feed-forward for en maalt disturbance.
# Eksempel: F21 Q20, disturbancebidraget er -D(s)*d(s)
# ideal_disturbance_feedforward([10.5, 21], [1, 4, 21], [1], [0.01, 1], -1)

# ---------------------------------------------------------------------------
# STABILITETSINTERVAL FOR ET PARAMETERAFHAENGIGT KARAKTERISTISK POLYNOMIUM
# ---------------------------------------------------------------------------
# find_stable_gain_ranges(characteristic, gain, s)
# Input: p(s,K)=0 eller en ligning som 1+K*G(s)=0, samt reelle K og s.
# Funktionen samler automatisk en broek og bruger taelleren som p(s,K).
# Metode: loeser p(j*w,K)=0 og tester polerne mellem de fundne graenser.
# Output: marginale (K,w)-punkter og aabne intervaller med alle poler i LHP.
# Graensepunkterne er IKKE asymptotisk stabile, fordi de har poler paa jw-aksen.
# Brug kun naar p(s,K) er udledt, og graden i s ikke aendrer sig med K.
#
# s, K = symbols("s K", real=True)
# p = (s + 1)**3 + K                 # Direkte polynomium
# p = 1 + K / (s + 1)**3            # Samme opgave som 1+K*G(s)=0
# stability = find_stable_gain_ranges(p, K, s)
# print("Marginale punkter (K, w):", stability["boundary_points"])
# print("Stabile K-intervaller:", stability["stable_gain_intervals"])
# print("Stabile intervaller for K > 0:", stability["positive_stable_gain_intervals"])
# For dette eksempel: -1 < K < 8, og hvis K skal vaere positiv: 0 < K < 8.
# Skift blot linjen med p = ... ud med karakteristisk polynomium fra din opgave.
#
# solve_stability_interval_by_boundary(char_poly, gain_symbol, variable=s)
# Input: symbolsk karakteristisk polynomium og en reel gain-parameter.
# Brug naar: samme j*w-metode oenskes med et kompakt outputdictionary.
# Eksempel: solve_stability_interval_by_boundary((s + 1)**3 + K, K, s)

In [4]:
G1 =4/(s+10)
G2 =1/(s+2)
G3 =(s+2)/(s+10)

G = (G1 + G2*G3)/(1-(G1 + G2*G3))

analyze_transfer_function(G)

{'G(s)': 5/(s + 5),
 'numerator': 5,
 'denominator': s + 5,
 'zeros_exact': {},
 'poles_exact': {-5: 1},
 'zeros_numeric': [],
 'poles_numeric': [(-5+0j)],
 'dc_gain': 1,
 'order': 1,
 'numerator_order': 0,
 'system_type': 0,
 'stable': True,
 'stability_text': 'asymptotisk stabil',
 'y(0+)': 0,
 'y(inf)': 1,
 'gain_crossover_frequency': None,
 'phase_at_gain_crossover_deg': None,
 'phase_margin_deg': None,
 'phase_margin_text': 'ingen gain crossover fundet'}